In [0]:
dbutils.widgets.text('SourceSystem', "")
dbutils.widgets.text('TableName', "")
dbutils.widgets.text('LoadType', "")
dbutils.widgets.text('DWRunID', "")

In [0]:
source_system = dbutils.widgets.get('SourceSystem') 
table_name = dbutils.widgets.get('TableName')
load_type = dbutils.widgets.get('LoadType')
dw_run_id = dbutils.widgets.get('DWRunID')

In [0]:
%run "../Library/DWFunctions"

In [0]:
df_bronze_configurations = read_delta_table('metadata.bronze.bronze_metadata_configurations')

In [0]:
root_path = df_bronze_configurations.select(col('RootPath')).first()[0]
file_format = df_bronze_configurations.select(col('FileFormat')).first()[0]

In [0]:
df_raw = read_file(f'{root_path}/{source_system}/{table_name}.{file_format}', file_format)

In [0]:
cols = df_raw.columns
cols_cleaned = []
for col in cols:
    col = col.strip().lower()
    col = col.replace(' ', '_')
    cols_cleaned.append(re.sub("[()-]", '', col))
cols_mapping =  {key: value for key,value in zip(cols,cols_cleaned)}

df_raw_cleaned = df_raw.withColumnsRenamed(cols_mapping) 

#df_raw.select(col.alias(cols_cleaned) for col in cols)

In [0]:
df_raw_cleaned = clean_column_names(df_raw)

In [0]:
df_bronze = (df_raw_cleaned
             .withColumn('DW_processing_time', lit(current_timestamp()))
             .withColumn('source_system', lit('Kaggle')))

In [0]:
sql_table_name = df_bronze_configurations.select("SQLTableName").first()[0]

In [0]:
write_dataframe_to_delta_table(df_bronze, f'bronze.{source_system}.{sql_table_name}')